# Setup and access check

Origami Health · synthetic training only. Website: setup. Planned block: 20 minutes.

Run selected prepared cells; do not provision services in class. Configure the full asset pack and runtime bindings before running. Live AIDP execution and classroom timing remain unverified.


## First-time user guide

**Learning objective:** Confirm you can use your assigned prepared environment before running a data pipeline.

**Before you begin:** Your facilitator has supplied your personal login, assigned workspace, notebook copy and participant slot. They have already prepared compute, the full asset pack and Lakehouse tables.

**Key terms:** A notebook mixes explanation (Markdown) with executable code cells. A session runs your Python code on assigned compute. A catalog groups data objects; a schema groups related tables. A volume holds files.

**Timebox:** This notebook belongs to the 20-minute website block. Read the explanations as you run the prepared cells. Optional exploration is not part of the core timebox.

**How to run:** There are 2 code cells below. Run them one at a time, from top to bottom, using the editor's run-cell control. Wait for completion, read the actual output and only then continue. Do not use Run all on your first pass. All runtime bindings are facilitator-prepared; do not paste credentials or edit another participant's paths.

For common problems, open START_HERE.md at the asset-pack root. If an assertion fails, stop and keep the error for the facilitator; do not change the assertion or expected values to make it pass.


## Load administrator-prepared participant bindings

**Purpose:** Load administrator-prepared participant bindings

**Inputs:** Full asset pack plus external ORIGAMI_ASSET_ROOT / ORIGAMI_RUNTIME_CONFIG bindings.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Assigned slot and synthetic snapshot; placeholders fail closed.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 1:** Run the binding cell. Check that the printed participant slot matches your assignment. Do not change the slot to gain access to another participant’s files.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
import os, sys, json
from pathlib import Path
asset_root = os.environ.get('ORIGAMI_ASSET_ROOT')
if not asset_root:
    raise RuntimeError('Facilitator must configure ORIGAMI_ASSET_ROOT and ORIGAMI_RUNTIME_CONFIG before class.')
sys.path.insert(0, str(Path(asset_root) / 'shared'))
from origami_core import AS_OF, SOURCES, fingerprint, score_rows
from aidp_runtime import configuration, bounded_rows, persist, read_delta, source_frame, verify_fixture, unique, publish_snapshot, save_evidence
cfg = configuration()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sparkContext.addPyFile(str(Path(asset_root) / 'shared' / 'origami_core.py'))
print('Participant:', cfg['participant_id'], '| synthetic snapshot:', AS_OF)

## Read-only readiness check

**Purpose:** Read-only readiness check

**Inputs:** Warm Spark, source CSVs and pre-created external tables.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** 725 source claims and target tables visible; separate live write/denial checks are still required.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 2:** Run the readiness cell. It reads the source and checks table visibility; it does not create infrastructure or prove write permissions.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
assert spark.range(1).count() == 1
raw = source_frame(spark,cfg,'claims')
assert raw.count() == 725
raw.select('source_record_id','claim_id','submitted_amount').show(3,truncate=False)
# Read-only table/schema presence check; no DDL, writes or permissions are changed.
for name in ['claim_detail','claims_monthly','member_plan','benefit_usage','benefit_schedule']:
    spark.table(f"{cfg['target_catalog']}.{cfg['target_schema']}.origami_{name}").limit(0).collect()
print('Basic read checks passed. Facilitator must separately verify assigned write permissions and cross-slot denial.')

## Completion checkpoint

- [ ] Every required code cell completed without error.
- [ ] I checked the actual result: You see the correct participant slot, synthetic snapshot 2026-09-01, three sample source rows and the message Basic read checks passed. No red error is present.
- [ ] I saved evidence: Record your slot, notebook name and readiness result in the evidence sheet. Do not record passwords, keys or authentication tokens.

**Discuss:** Which checks confirm basic access, and which permission checks still belong to the facilitator?

**Self-check explanation:** A Spark query, the 725-row CSV and target-table reads confirm basic access. They do not prove write permission, participant isolation or that an agent enforces member scope.

**Next:** day1/01_bronze.ipynb. Continue only after the core checkpoint passes, or after the facilitator explicitly records a labelled fallback.

## If you get stuck

Note the notebook name, cell heading and sanitized error. Missing configuration, permission errors and missing target tables are facilitator setup issues, not instructions to provision resources. Unexpected values are validation failures: do not erase tables, disable checks or rerun the entire pipeline blindly. See START_HERE.md for the troubleshooting table.
